# Fine Tuning A Chatbot for Medical questions:
Developing a medical chat system using medical Q and A data.

## Part 1: 
### 1a. Data Exploration
I have a dataset in a csv with common medical questions and answers. I will not be comitting it to my repo as it is a large file. The purpoe of this project is to highlight my strategy and approach. If you would like to replicate my project, modify the code to align with your own data.s

In [ ]:
import pandas as pd
import jsonlines

# read in the data
med_data = pd.read_csv("data/med-data.csv")
# quick glance at the data
med_data.head()

,question,answer
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...
1,What is (are) Glaucoma ?,The optic nerve is a bundle of more than 1 mil...
2,What is (are) Glaucoma ?,Open-angle glaucoma is the most common form of...
3,Who is at risk for Glaucoma? ?,Anyone can develop glaucoma. Some people are a...
4,How to prevent Glaucoma ?,"At this time, we do not know how to prevent gl..."


In [27]:
# check the data size: (rows, columns)
med_data.shape

(16406, 2)

In [28]:
# of the questions, how may are unique?
len(med_data['question'].unique())


14981

In [29]:
# if we were to drop the duplicate rows, how much data would remain?
unique_data = med_data.drop_duplicates()
unique_data.shape

(16358, 2)

We have found that the data set has two columns: questions and answers. We have 16,406 rows of data. There are some duplicate questions-- only 14,981 unique questions and 16,358 unique overall rows. This means that some questions appear multiple times with different answers. I am comfortable with this since chat conversations are expected to be non-deterministic. There are also 48 duplicate rows in the dataset. This is a relatively small amount, and since I don't know the full origin of my data set, the duplication may be representative of the actual frequency of certain question / answer pairs. I will leave the duplicates in for now.

A cursory glance also shows that there are questions that are slight variations of other questions as well. This variation is good. It will likely help our model handle real life variation as well.

Let's check for missing data.

In [30]:
rows_with_missing_data = med_data[med_data.isnull().any(axis=1)]
rows_with_missing_data


,question,answer
3587,What is (are) HELLP syndrome ?,NaN
3836,What is (are) X-linked lymphoproliferative syn...,NaN
4196,What is (are) Familial HDL deficiency ?,NaN
4429,What is (are) Emery-Dreifuss muscular dystroph...,NaN
6689,What is (are) Emery-Dreifuss muscular dystroph...,NaN


Not bad. I will go ahead and drop these rows, since they won't be useful for our model.

In [31]:
cleaned_data = med_data.dropna()

In [32]:
cleaned_data.shape

(16401, 2)

### 1b: Data processing

For my first model attempt, I want to fine tune a model using instruction fine tuning. I'm choosing instruction fine tuning because it combines the power of the cutting edge larget language models with the specificity of our use-case data set. We also seem to have a good amount of data for this use case, but not so much that we would be able to match the cability of the big LLMs that have taken a lot of resources and data to train.

I plan to use hugging face libraries and will feed the med data into an input output format that can be saved a jsonlinesfile.


In [41]:
# reformat to create a datalines file (ideal for hugging face SRT)
processed_med_qa = []

for index, row in cleaned_data.iterrows():
    processed_med_qa.append({"input": row["question"], "output": row["answer"]})


In [ ]:
# confirm we have the expected number of QA pairs
len(processed_med_qa)



16401

In [44]:
# check the format is what we would expect
processed_med_qa[0]

{'input': 'What is (are) Glaucoma ?',
 'output': "Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. The most common form of the disease is open-angle glaucoma. With early treatment, you can often protect your eyes against serious vision loss. (Watch the video to learn more about glaucoma. To enlarge the video, click the brackets in the lower right-hand corner. To reduce the video, press the Escape (Esc) button on your keyboard.)  See this graphic for a quick overview of glaucoma, including how many people it affects, whos at risk, what to do if you have it, and how to learn more.  See a glossary of glaucoma terms."}

In [ ]:

with jsonlines.open(f'med_chat_processed.jsonl', 'w') as writer:
    writer.write_all(processed_med_qa)
    

I have decided to fine tune with the llama models. I will need to reformat my data accordingly. First I will pull the model information from hugging face.

In [ ]:
import os
# note that I created a fine-grained token via the hugging face website
# you will need to accept the terms for any model you would like to use as well.
TOKEN = os.environ["HUGGINGFACE_HUB_TOKEN"]

For this application, I considered using a model from the huggingface leader board as my base, but utimately decided to go with the llama models because:
- I am familiar with them
- I know they would work well with this task
- I am confident they will be able to run given my resource constraints. I do not have a personal GPU-- I will be running on Google Collab.

I've decided to go with `Llama-3.1-8B-Instruct` because it works well for chat and the size is reasonable given my limitations. According to the team, "The Llama 3.1 instruction tuned text only models (8B, 70B, 405B) are optimized for multilingual dialogue use cases and outperform many of the available open source and closed chat models on common industry benchmark" see  https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct.

In [ ]:
from transformers import AutoTokenizer
import json
from transformers import AutoModelForCausalLM, AutoTokenizer

# pull the model information

model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-LLama-3.1-8B-Instruct", token=TOKEN)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-LLama-3.1-8B-Instruct", token=TOKEN)

with open("med_chat_processed.jsonl", "r", encoding="utf-8") as f, open("out2.jsonl", "w", encoding="utf-8") as fout:
  for line in f:
     ex = json.loads(line)
     messages = [
         {"role": "user", "content": ex["input"]},
         {"role": "assistant", "content": ex["output"]}
     ]
     prompt = tokenizer.apply_chat_template(
         messages,
         tokenize=False,
         add_generation_prompt=False
     )
     fout.write(json.dumps({"text": prompt}, ensure_ascii=False)+ "\n")

The next phase of our project must be done in an environment with a GPU. I have created a Google Colab instance using the A100 chip and the High-Ram option toggled on. Continue to Part2-MedChatBot-Training.ipynb, which was run in Colab.